<a href="https://colab.research.google.com/github/davidmkidd/UK-Supermarket-Carbon-Emissions/blob/main/UKSmktComp_Emissions_Scope1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://evoviz.uk/wp-content/uploads/2026/04/Food_Divider_trans2.png">

# Scope 1

Supermarket Scope 1 emissions are the sum of four emission categories.

$C_{scope1} = C_{gas} + C_{transport} + C_{fugitive} + C_{other}$

Only location-based Scope 1 emissions are reported as there are just two market-based values.

Retailer emissions are plotted by year, modelled linearly with store number and area, and scaled to intensity.

# Set-up

## Libraries and Data

In [ ]:
# Load libraries
library(dplyr)   # Data manipulation
library(ggplot2) # Graphing
library(repr)    # Graph size
options(repr.plot.width = 10, repr.plot.height = 8)
library(broom)   # Format model output
library(knitr)    # Format model output

# Download Summarised Emmissions Data
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/retailer_emissions_yr.csv"
download_path <- "/content/retailer_emissions_yr.csv"
download.file(url, destfile = download_path, mode = "wb")
emissions.yr <- read.csv("/content/retailer_emissions_yr.csv", header=TRUE, stringsAsFactors=FALSE)

# Download Retailer Data
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/retailer_data.csv"
download_path <- "/content/retailer_data.csv"
download.file(url, destfile = download_path, mode = "wb")
retailer.data <- read.csv("/content/retailer_data.csv", header=TRUE, stringsAsFactors=FALSE)

# Make palette list
retailer.pal <- setNames(retailer.data$hex, retailer.data$retailer_code)

# Make code/name list
retailer.code <- setNames(retailer.data$retailer, retailer.data$retailer_code)

## Named lists

Make retailer-hex code and retailer_code-retailer key-pair named lists from retailer attribute table.

In [ ]:
# Make palette list
retailer.pal <- setNames(retailer.data$hex, retailer.data$retailer_code)

# Make code/name list
retailer.code <- setNames(retailer.data$retailer, retailer.data$retailer_code)

## Year Truncation

Reduce number of year characters to two to reduce plot space occupied by year labels.

In [ ]:
# Year as two numerals for plotting
emissions.yr$year2 <- emissions.yr$year - 2000

## Split Absolute and Intenstity KPIs

Split absolute and intenstity values into seperate data frames as they will be used differently.


In [ ]:
# Reported Absolute Values
emissions.yr.absolute <- emissions.yr %>%
   filter(kpi_type == "Measure")
nrow(emissions.yr.absolute)

# Reported Intensity Values
emissions.yr.intensity <- emissions.yr %>%
   filter(kpi_type == "Intensity")
nrow(emissions.yr.intensity)

* More absolute values are reported than intensity metrics.
* Intensity metrics were not required until 2019 SECR.

# Absolute Emissions

Plot Scope 1 emissions though time.

In [ ]:
# Get Absolute Scope 1 location-based emissions and order
data.abs <-  emissions.yr.absolute %>%
  filter(kpi == "Scope 1" & kpi_type == "Measure" & method == "Location") %>%
  arrange(retailer_code, year)
nrow(data.abs)

In [ ]:
# Plot Absolute Scope 1 location-based emissions
ggplot(data.abs, aes(x = year, y = value/1000, colour = retailer_code)) +
  geom_path(na.rm = TRUE)  +
  geom_point(na.rm = TRUE) +
  ggtitle("Reported Scope 1 Emissions") +
  xlab("Year") +
  ylab("1000 tCOe") +
  scale_x_continuous(limits = c(2005, 2025)) +
  scale_colour_manual(name = "Retailer", values = retailer.pal, labels = retailer.code) +
  theme_classic()

What pattern does this plot show, and how should it be interpreted and acted upon?

* The compeateness of Scope 1 emission records varies from 14 data points (Tesco) to just two (Asda).

* Four retailers report baseline values for 2005 or 2006 followed by a >10 year gap before values are annually reported.

* Other retailer began reporting in 2012 and 2019.

* The 'big four' have been the highest emitters with Tesco responsible for twice the emissions of other retailers.

* The emissions of all retailers except Marks and Spencer and Ocado show long-term decline.

* Ocado's increase may reflect their market growth and the ownership of carbon-fuelled vehicles.

* The recent rise in M&S emissions requies further investigation.




Scope 1 emissions are expected to increase with retailer size, estimated here as store number and area.



## Store Number

Plot emissions against store number.

In [ ]:
# Plot Reported Values Against Store Number
ggplot(data.abs, aes(x = total_store, y = value/1000, colour = retailer_code, label = year2)) +
  geom_text(na.rm = TRUE) +
  ggtitle("Reported Scope 1 Emissions by Store Number") +
  xlab("Store Number") +
  ylab("1000 tCOe") +
  scale_x_continuous(limits = c(0, 3000)) +
  scale_colour_manual(name = "Retailer", values = retailer.pal, labels = retailer.code) +
  theme_classic()

* As expected, retailers with more stores have greater scope 1 emissions.

* The COOP, Aldi, Lidl and Iceland appear to have relatively low emissions for their store number.

The strength of the relationship between emissions and retailer size is tested with linear regression, but:

1.  Yearly retailer values are temporally correlated, breaking the assumptions of linear regression.

2. The number of retailers reporting varies by year.

In [ ]:
# Retailers reporting Scope 1 emissions by year
table(data.abs$year)

3. The emissions reported by a retailer may reflect 'anomalous' business conditions and so not be representative of 'normal business'.

To address these issues, mean emission from 2020 to 2022 are regressed against store number.

In [ ]:
# Mean emissions for years 2020 - 2022
data.abs.mean <- data.abs %>%
  filter(year >= 2020 & year <= 2022) %>%
  group_by(retailer_code) %>%
  summarise_at(vars(value, total_store, total_area),list(mean = mean))

In [ ]:
data.abs.mean <- data.abs.mean %>%
  rename(value = value_mean, total_store = total_store_mean, total_area = total_area_mean)

names(data.abs.mean)

In [ ]:
# Linear Regression Scope 1 ~ Store Number
lm_store <- lm(data = data.abs.mean, value ~ total_store, na.action=na.omit)
glance(lm_store) %>%
  kable(digits = 3)
tidy(lm_store) %>%
  kable(digits = 3)

print(ggplot(data.abs.mean, aes(x = value/1000, y = total_store)) +
  geom_point(na.rm = TRUE) +
  geom_smooth(method='lm', na.rm = TRUE) +
  ggtitle("Scope 1 Emissions and Mean Store Number 2020 - 2022") +
  xlab("Mean Emissions 1000 tCO2e") +
  ylab("Mean Store Number"))

* Store number predicts 49.2% of Scope 1 emissions with *p* = 0.024.

## Store Area

In [ ]:
# Plot Reported Values Against Store Area
ggplot(data.abs, aes(x = total_area/1000, y = value/1000, colour = retailer_code, label = year2)) +
  geom_text(na.rm = TRUE) +
  ggtitle("Reported Scope 1 Emissions by Store Area") +
  xlab("Store Area m2") +
  ylab("1000 tCOe") +
  scale_x_continuous(limits = c(0, 2500)) +
  scale_colour_manual(name = "Retailer", values = retailer.pal, labels = retailer.code) +
  theme_classic()

* Scope 1 emissions appears to have a stronger linear relationship with total retailer store area than store number.

* Tesco has slightly higher emissions for its store area than other retailers.

* Aldi, Iceland, and Lidl have slightly lower emissions for their store area than other retailers.

In [ ]:
# Scope 1 ~ Area
lm_area <- lm(data = data.abs.mean, value ~ total_area, na.action=na.omit)
glance(lm_area) %>%
  kable(digits = 3)
tidy(lm_area) %>%
  kable(digits = 3)

ggplot(data.abs.mean, aes(x = value/1000, y = total_area/1000)) +
  geom_point(na.rm = TRUE) +
  geom_smooth(method='lm', na.rm = TRUE) +
  ggtitle("Scope 1 Emissions and Mean Store Area 2020 - 2022") +
  xlab("Mean Emissions 1000 tCO2e") +
  ylab("Mean Store Area 1000 m2")

* Store area explains 85.9% of Scope 1 emissions with *p* = < 0.001, so is a better model than store number.

## Store Number and Area


In [ ]:
# Linear Regression Scope 1 ~ Mean Store Number and Mean Store Area 2000 or later
lm_both <- lm(data = data.abs.mean, value ~ total_store + total_area, na.action=na.omit)
glance(lm_both) %>%
  kable(digits = 3)
tidy(lm_both) %>%
  kable(digits = 3)

* Both variables explain 90.6% of variation in Scope 1 emissions with *p* < 0.001, but store number is not significant with *p* < 0.102.

## Best Model

In [ ]:
anova(lm_store, lm_both)

In [ ]:
anova(lm_area, lm_both)

The best model is,

$C_{scope1} \sim StoreArea$

as *lm_both* is significantly better than *lm_store*, but not significantly better than *lm_area*.

# Emission Intensity
Estimate Scope 1 emissions store area-based intensity.


In [ ]:
# Calculate estimated intensity for absolute values
data.abs$intensity <- data.abs$value/data.abs$total_area

Are there any reported Scope 1 area-based intensity metrics to which estimated values can be compared?

In [ ]:
# Reported Values
data.int <- emissions.yr.intensity %>%
  filter(kpi == "Scope 1" & kpi_type == "Intensity" & method == "Location" & unit_denom_1 == "m2")
nrow(data.int)

There are no reported intensity metrics so just plot estimated intensity.

In [ ]:
# Plot Estimated intensity
ggplot(data.abs, aes(x = year, y = intensity, colour = retailer_code)) +
  geom_line(na.rm = TRUE)  +
  geom_point(na.rm = TRUE) +
  ggtitle("Estimated Scope 1 Intensity") +
  xlab("Year") +
  ylab(expression(paste("tCO"[2],"e/1000m"^2))) +
  scale_colour_manual(name = "Retailer", values = retailer.pal, labels = retailer.code) +
  scale_x_continuous(limits = c(2014, 2023), breaks=seq(2014,2023)) +
  theme_classic()

* All retailers except M&S show long-term reduction in internsity.
* Tesco remains the highest emitter but the difference with other retailers is reduced.
* The COOP was the second highest emitter but reduced its intensity to the mean.
* Low price Aldi, Iceland, and Lidl have lower Scope 1 intensities than other retailers.
* The recent rise in Marks and Spencer emissions is more noticable.
* OCADO has no stores so has no area-based intensity metric.

---

[Main Page](https://colab.research.google.com/drive/1f8a0pXfF9PqCujiwjf4TO4-k7ezt-6b3?usp=sharing)

[Scope 2 Location-based](https://colab.research.google.com/drive/1fo6NJ-2t7_VVn99Js8fanx3dGDiVdZud?usp=sharing)

---
